In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mstats
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       926Mi       5.9Gi       2.0Mi       5.8Gi        11Gi
Swap:             0B          0B          0B


# Phase 1 Data Acquistion Joining and cleaning
## Question 1
Load all nine CSV files using pd.read_csv(). For files exceeding available RAM, use chunksize appropriately. Save the concatenated output as a Parquet file and report: (i) memory usage before and after downcasting, and (ii) file size of the resulting Parquet


Loading data

In [3]:
os.chdir('/content/drive/MyDrive/EDA/')

In [4]:
os.listdir('/content/drive/MyDrive/EDA/')

['loans_master.csv',
 'customer_bureau.csv',
 'loan_enquiry_bureau.csv',
 'monthly_emi_track.csv',
 'collateral_assets.csv',
 'payment_history.csv',
 'loan_performance.csv',
 'credit_card_behavior.csv',
 'branch_region_economy.csv',
 'payment_history.parquet',
 'loan_performance.parquet',
 'branch_region_economy.parquet',
 'credit_card_behavior.parquet',
 'monthly_emi_track.parquet',
 'collateral_assets.parquet',
 'loan_enquiry_bureau.parquet',
 'customer_bureau.parquet',
 'loans_master.parquet']

In [5]:
path = "/content/drive/MyDrive/EDA"
csv_files = []
for file in os.listdir(path):
    if file.endswith(".csv"):
        csv_files.append(file)

all_df = {}
for file in csv_files:
    df = pd.read_csv(path + "/" + file)
    all_df[file] = df

In [6]:
all_df['loan_performance.csv'].head()

,loan_id,dpd_bucket,npa_flag,risk_tier,cibil_score_band,recoveries_inr,recovery_fee_rate,policy_code,hardship_amount_inr,hardship_length_mths,net_loss_inr,provision_inr
0,LN000000001,Current,0,Medium Risk,Poor,0.0,0.0,1,0.00,0,0.0,397.93
1,LN000000002,Current,0,Low Risk,Excellent,0.0,0.0,1,11568.37,9,0.0,304.07
2,LN000000003,Current,0,Medium Risk,Poor,0.0,0.0,1,0.00,0,0.0,483.40
3,LN000000004,Current,0,High Risk,Excellent,0.0,0.0,1,0.00,0,0.0,4730.75
4,LN000000005,Current,0,Low Risk,Poor,0.0,0.0,1,0.00,0,0.0,379.61


In [7]:
for name, df in all_df.items():
    print(f"{name}: {df.shape}")

loans_master.csv: (2000000, 27)
customer_bureau.csv: (2000000, 30)
loan_enquiry_bureau.csv: (2000000, 24)
monthly_emi_track.csv: (2000000, 23)
collateral_assets.csv: (2000000, 20)
payment_history.csv: (2000000, 18)
loan_performance.csv: (2000000, 12)
credit_card_behavior.csv: (2000000, 17)
branch_region_economy.csv: (2000000, 19)


Checking memory before downcasting

In [8]:
memory_before = sum(df.memory_usage(deep=True).sum() for df in all_df.values()) / (1024**2)
print(f"Memory BEFORE downcasting: {memory_before:.2f} MB\n")

for file, df in all_df.items():

    mem = df.memory_usage(deep=True).sum()

    print(file, "=", round(mem/(1024**2),2), "MB")

Memory BEFORE downcasting: 7524.14 MB

loans_master.csv = 1564.73 MB
customer_bureau.csv = 1014.09 MB
loan_enquiry_bureau.csv = 1109.23 MB
monthly_emi_track.csv = 808.17 MB
collateral_assets.csv = 844.51 MB
payment_history.csv = 373.84 MB
loan_performance.csv = 558.29 MB
credit_card_behavior.csv = 587.13 MB
branch_region_economy.csv = 664.13 MB


Downcasting the dataframes

In [9]:
import pandas as pd

def reduce_memory(df):

    for col in df.columns:

        if df[col].dtype == "int64":                 #downcasting int columns

            df[col] = pd.to_numeric(
                df[col],
                downcast="integer"
            )

        elif df[col].dtype == "float64":              #downcasting float columns

            df[col] = pd.to_numeric(
                df[col],
                downcast="float"
            )

        elif df[col].dtype == "object":             #downcasting object coluns to string

            unique_values = df[col].nunique()

            if unique_values < 100:

                df[col] = df[col].astype("category")

    return df

In [10]:
for file in all_df:

    all_df[file] = reduce_memory(all_df[file])

Cheking memory after downcasting

In [11]:
memory_after = 0

for file, df in all_df.items():

    mem = df.memory_usage(deep=True).sum()

    memory_after += mem

    print(file, "=",
          round(mem/(1024**2),2),"MB")

print(f"\nMemory BEFORE downcasting: {memory_before:.2f} MB")

print("\nTotal Memory After Downcasting =",
      round(memory_after/(1024**2),2),"MB")

loans_master.csv = 343.33 MB
customer_bureau.csv = 528.34 MB
loan_enquiry_bureau.csv = 288.01 MB
monthly_emi_track.csv = 379.56 MB
collateral_assets.csv = 308.99 MB
payment_history.csv = 299.45 MB
loan_performance.csv = 194.55 MB
credit_card_behavior.csv = 190.74 MB
branch_region_economy.csv = 308.99 MB

Memory BEFORE downcasting: 7524.14 MB

Total Memory After Downcasting = 2841.97 MB


Converting CSV to parquet

In [12]:
for file, df in all_df.items():

    parquet_name = file.replace(".csv", ".parquet")

    df.to_parquet(
        path + "/" + parquet_name,
        index=False
    )

In [13]:
for file in os.listdir(path):
    if file.endswith(".parquet"):
        print(file)

payment_history.parquet
loan_performance.parquet
branch_region_economy.parquet
credit_card_behavior.parquet
monthly_emi_track.parquet
collateral_assets.parquet
loan_enquiry_bureau.parquet
customer_bureau.parquet
loans_master.parquet


In [14]:
for file in os.listdir(path):

    if file.endswith(".parquet"):

        size = os.path.getsize(path + "/" + file)

        print(
            file,
            "=",
            round(size/(1024**2), 2),
            "MB"
        )

payment_history.parquet = 73.0 MB
loan_performance.parquet = 24.35 MB
branch_region_economy.parquet = 45.5 MB
credit_card_behavior.parquet = 45.02 MB
monthly_emi_track.parquet = 75.03 MB
collateral_assets.parquet = 30.73 MB
loan_enquiry_bureau.parquet = 40.93 MB
customer_bureau.parquet = 103.24 MB
loans_master.parquet = 64.21 MB


## Question 2
(b)	Join all nine tables on loan_id using sequential left merges. After every individual join, assert that the running row count equals 2,000,000. Report the number of orphan records found, if any, and explain what orphan records indicate about data integrity.

In [15]:
final_df = all_df['loans_master.csv'].copy()  # Start with master base table
print(f"Initial Shape: {final_df.shape}")  # Print rows and columns

final_df = pd.merge(final_df, all_df['loan_performance.csv'], on='loan_id', how='left')  # Merge table 2
print(f"Post Merge 1 Shape: {final_df.shape}")  # Print rows and columns

final_df = pd.merge(final_df, all_df['credit_card_behavior.csv'], on='loan_id', how='left')  # Merge table 3
print(f"Post Merge 2 Shape: {final_df.shape}")

final_df = pd.merge(final_df, all_df['collateral_assets.csv'], on='loan_id', how='left')  # Merge table 4
print(f"Post Merge 3 Shape: {final_df.shape}")

final_df = pd.merge(final_df, all_df['loan_enquiry_bureau.csv'], on='loan_id', how='left')  # Merge table 5
print(f"Post Merge 4 Shape: {final_df.shape}")

final_df = pd.merge(final_df, all_df['customer_bureau.csv'], on='loan_id', how='left')  # Merge table 6
print(f"Post Merge 5 Shape: {final_df.shape}")

final_df = pd.merge(final_df, all_df['payment_history.csv'], on='loan_id', how='left')  # Merge table 7
print(f"Post Merge 6 Shape: {final_df.shape}")

final_df = pd.merge(final_df, all_df['monthly_emi_track.csv'], on='loan_id', how='left')  # Merge table 8
print(f"Post Merge 7 Shape: {final_df.shape}")

final_df = pd.merge(final_df, all_df['branch_region_economy.csv'], on='loan_id', how='left')  # Merge table 9
print(f"Final Merged Shape: {final_df.shape}")

Initial Shape: (2000000, 27)
Post Merge 1 Shape: (2000000, 38)
Post Merge 2 Shape: (2000000, 54)
Post Merge 3 Shape: (2000000, 73)
Post Merge 4 Shape: (2000000, 96)
Post Merge 5 Shape: (2000000, 125)
Post Merge 6 Shape: (2000000, 142)
Post Merge 7 Shape: (2000000, 164)
Final Merged Shape: (2000000, 182)


In [16]:
#checking orphan count
master_ids = set(final_df["loan_id"])

count_of_orphans = 0

for file, df in all_df.items():

    if file != "loans_master.csv":

        total_orphans = (~df["loan_id"].isin(master_ids)).sum()

        print(file, "Orphan Records =", total_orphans)

        count_of_orphans += total_orphans

print("\nTOTAL ORPHANS =", count_of_orphans)

customer_bureau.csv Orphan Records = 0
loan_enquiry_bureau.csv Orphan Records = 0
monthly_emi_track.csv Orphan Records = 0
collateral_assets.csv Orphan Records = 0
payment_history.csv Orphan Records = 0
loan_performance.csv Orphan Records = 0
credit_card_behavior.csv Orphan Records = 0
branch_region_economy.csv Orphan Records = 0

TOTAL ORPHANS = 0



Orphan records are rows in secondary child tables whose `loan_id` cannot be found anywhere within the primary parent table (`loans_master.csv`). These records indicate a breakdown in **Referential Integrity**, which typically happens when data is extracted at different times across systems or when foreign key database constraints are missing. Because a sequential `left merge` is used to maintain the exact 2,000,000 row boundary, these orphan records are automatically dropped from the final analytical dataset.

In [17]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Columns: 182 entries, loan_id to poor_monsoon_year_flag
dtypes: category(37), float32(55), float64(29), int16(3), int32(1), int8(49), object(8)
memory usage: 1.1+ GB


In [18]:
final_df.columns

Index(['loan_id', 'issue_date', 'issue_year', 'issue_month', 'loan_amnt_inr',
       'funded_amnt_inr', 'loan_term_months', 'int_rate_pct',
       'installment_inr', 'annual_installment_inr',
       ...
       'branch_age_years', 'branch_size', 'loan_officer_exp_years',
       'branch_npa_rate', 'branch_sanction_rate', 'infrastructure_index',
       'financial_inclusion_idx', 'credit_penetration_idx',
       'covid_issue_year_flag', 'poor_monsoon_year_flag'],
      dtype='object', length=182)

##Question 1(c)
The dataset contains eight deliberately injected data quality issues across multiple columns. Identify all eight, create a binary dirty_flag column to mark affected rows, and for each issue state: the column affected, the approximate count of dirty records, why the value is invalid, and the imputation strategy you applied.

In [19]:
# Checking for missing values
rd_summary = pd.DataFrame({
    'Missing Count': final_df.isnull().sum(),         #missing record count
    'Percentage (%)': final_df.isnull().mean() * 100  # missing percentange
})

print(rd_summary[rd_summary['Missing Count'] > 0].sort_values(by='Percentage (%)', ascending=False))

                           Missing Count  Percentage (%)
vehicle_type                     1879789        93.98945
property_city_tier               1679672        83.98360
property_type                    1679672        83.98360
mths_since_last_record           1598332        79.91660
ltv_ratio_pct                    1472196        73.60980
valuation_agency                 1370048        68.50240
charge_type                      1258896        62.94480
collateral_type                  1258896        62.94480
mths_since_last_delinq           1098640        54.93200
primary_cc_bank                   560528        28.02640
primary_card_type                 560528        28.02640
top_spend_category                560528        28.02640
income_doc_type                   308704        15.43520
collateral_score                  298379        14.91895
mort_acc                          260330        13.01650
il_util_pct                       200167        10.00835
emp_length_years               

In [20]:
df_clean = final_df.copy()  # Create a  copy of final_df
df_clean['dirty_flag'] = 0  # Initialize binary dirty_flag to 0 for all records
print("Clean copy created and dirty_flag initialized.")

Clean copy created and dirty_flag initialized.


In [21]:

print("Income columns:", [c for c in final_df.columns if 'inc' in c.lower()])
print("CIBIL columns:", [c for c in final_df.columns if 'cibil' in c.lower()])
print("Rate columns:", [c for c in final_df.columns if 'rate' in c.lower()])
print("Employment columns:", [c for c in final_df.columns if 'emp' in c.lower()])
print("DTI columns:", [c for c in final_df.columns if 'dti' in c.lower()])
print("Utilization columns:", [c for c in final_df.columns if 'util' in c.lower()])
print("Account columns:", [c for c in final_df.columns if 'acc' in c.lower()])
print("Year columns:", [c for c in final_df.columns if 'year' in c.lower()])

Income columns: ['income_doc_type', 'annual_inc_inr', 'mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_pymnt', 'emi_to_income_ratio', 'state_per_capita_inc_inr', 'financial_inclusion_idx']
CIBIL columns: ['cibil_score_band', 'cibil_score']
Rate columns: ['int_rate_pct', 'rbi_repo_rate_pct', 'rate_spread_pct', 'real_interest_rate_pct', 'recovery_fee_rate', 'rejection_rate_pct', 'state_literacy_rate_pct', 'branch_npa_rate', 'branch_sanction_rate']
Employment columns: ['emp_title', 'emp_length_years']
DTI columns: ['dti_pct']
Utilization columns: ['cc_utilization_pct', 'revol_util_pct', 'bc_util_pct', 'il_util_pct', 'all_util_pct']
Account columns: ['open_acc', 'total_acc', 'mort_acc', 'num_accts_ever_120_pd', 'acc_now_delinq', 'ots_accepted_flag']
Year columns: ['issue_year', 'property_age_years', 'vehicle_age_years', 'emp_length_years', 'credit_hist_years', 'branch_age_years', 'loan_officer_exp_years', 'covid_issue_year_flag', 'poor_monsoon_year_flag']


In [22]:
# Issue 1: Negative Income
issue1 = df_clean['annual_inc_inr'] < 0  # Identify negative annual income rows
print(f"Issue 1 | Affected Column: annual_inc_inr       | Dirty Count: {issue1.sum()}")  # Print identification metrics
df_clean.loc[issue1, 'dirty_flag'] = 1  # Update binary tracking column to 1
df_clean.loc[issue1, 'annual_inc_inr'] = df_clean['annual_inc_inr'].median()  # Impute invalid values with column median

Issue 1 | Affected Column: annual_inc_inr       | Dirty Count: 0


In [23]:
# Issue 2: Invalid CIBIL Scores
issue2 = (df_clean['cibil_score'] < 300) | (df_clean['cibil_score'] > 900)  # Identifying outliers if any
print(f"Issue 2 | Affected Column: cibil_score          | Dirty Count: {issue2.sum()}")  # print sum of outliers
df_clean.loc[issue2, 'dirty_flag'] = 1  # Track anomaly in binary flag column
df_clean.loc[issue2, 'cibil_score'] = df_clean['cibil_score'].median()  # Impute invalid entries with column median

Issue 2 | Affected Column: cibil_score          | Dirty Count: 0


In [24]:
#Issue 3: Negative Interest Rates
issue3 = df_clean['int_rate_pct'] < 0  # Check for impossible retail interest rates less than 0%
print(f"Issue 3 | Column: int_rate_pct          | Dirty Count: {issue3.sum()}")  # Print total count of rows matching this anomaly
df_clean.loc[issue3, 'dirty_flag'] = 1  # Set binary dirty flag to 1 for all matching invalid rows
df_clean.loc[issue3, 'int_rate_pct'] = df_clean['int_rate_pct'].median()  # Impute invalid interest rates using the column median value

Issue 3 | Column: int_rate_pct          | Dirty Count: 0


In [26]:
# Issue 4: Employment Length years
issue4 = df_clean['emp_length_years'] > 60  # Check for working tenures exceeding a realistic 60-year lifespan barrier
print(f"Issue 4 | Column: emp_length_years      | Dirty Count: {issue4.sum()}")  # Print total count of rows matching this anomaly
df_clean.loc[issue4, 'dirty_flag'] = 1  # Set binary dirty flag to 1 for all matching invalid rows
df_clean.loc[issue4, 'emp_length_years'] = df_clean['emp_length_years'].mode()[0]  # Impute invalid lengths using the most frequent value (mode)

Issue 4 | Column: emp_length_years      | Dirty Count: 0


In [27]:
#Issue 5: Negative DTI Ratios
issue5 = df_clean['dti_pct'] < 0  # Check for impossible negative Debt-to-Income percentage metrics
print(f"Issue 5 | Column: dti_pct               | Dirty Count: {issue5.sum()}")  # Print total count of rows matching this anomaly
df_clean.loc[issue5, 'dirty_flag'] = 1  # Set binary dirty flag to 1 for all matching invalid rows
df_clean.loc[issue5, 'dti_pct'] = df_clean['dti_pct'].median()  # Impute invalid DTI profiles using the column median value

Issue 5 | Column: dti_pct               | Dirty Count: 0


In [28]:
#Issue 6:Revolving Credit Utilization

# Check if credit card usage is greater than 500% (which is an impossible calculation error)
issue6 = df_clean['revol_util_pct'] > 100

# Print the total number of broken rows found with this issue
print(f"Issue 6 | Column: revol_util_pct        | Dirty Count: {issue6.sum()}")

# Mark these broken rows as 1 (dirty) in our tracking column
df_clean.loc[issue6, 'dirty_flag'] = 1

# Fix the error by capping the value at 100% (meaning the credit card is completely maxed out)
df_clean.loc[issue6, 'revol_util_pct'] = 100.0

Issue 6 | Column: revol_util_pct        | Dirty Count: 0


In [30]:
#Issue 7: Impossible Credit Account Logic
issue7 = df_clean['open_acc'] > df_clean['total_acc']  # Check if open active accounts outnumber lifetime historical accounts
print(f"Issue 7 | Column: open_acc vs total     | Dirty Count: {issue7.sum()}")  # Print total number of broken rows found
df_clean.loc[issue7, 'dirty_flag'] = 1  # Mark these broken rows as 1 (dirty) in our tracking column
df_clean.loc[issue7, 'open_acc'] = df_clean.loc[issue7, 'total_acc']  # Fix relationship mismatch by capping open accounts to equal total accounts

Issue 7 | Column: open_acc vs total     | Dirty Count: 0


In [29]:
# Issue 8: Future-Dated Loan Originations

# Check if the loan issue year is greater than 2026 (cannot issue loans in the future)
issue8 = df_clean['issue_year'] > 2026

# Print the total number of broken rows found with this issue
print(f"Issue 8 | Column: issue_year            | Dirty Count: {issue8.sum()}")

# Mark these broken rows as 1 (dirty) in our tracking column
df_clean.loc[issue8, 'dirty_flag'] = 1

# Fix the error by capping the year at 2026 (setting it back to the current real-world year)
df_clean.loc[issue8, 'issue_year'] = 2026

Issue 8 | Column: issue_year            | Dirty Count: 0


## Question 1
(d)	Classify the missing-value pattern for each high-missing column (mths_since_last_delinq, mort_acc, emp_length_years, il_util_pct) as MCAR, MAR, or MNAR. Justify each classification using either Little's MCAR test output or domain reasoning. Apply the correct imputation strategy and verify with .isnull().sum() before and after.



**mths_since_last_delinq**

* **Classification:** MNAR
* **Justification:** Missing values indicate perfect borrowers who have never been delinquent. The missingness itself carries critical risk information.
* **Imputation:** Filled with `-1` to represent "no delinquency history".

**il_util_pct**

* **Classification:** MNAR
* **Justification:** Missing values occur because the borrower simply has no active installment loan accounts, making utilization not applicable.
* **Imputation:** Filled with `0.0` to represent zero utilization.

**mort_acc**

* **Classification:** MAR
* **Justification:** Missing values typically correspond to borrowers who do not hold any mortgage accounts or have a shallow credit file.
* **Imputation:** Filled with column median.

**emp_length_years**

* **Classification:** MAR
* **Justification:** Missing employment lengths depend on observed borrower traits like age, income bracket, or specific application streams.
* **Imputation:** Filled with column mode (most frequent tenure).

In [31]:
#Print MAR missing counts BEFORE imputation
print("=== MAR COUNTS BEFORE ===")  # Section header
print(f"emp_length_years missing: {df_clean['emp_length_years'].isnull().sum()}")  # Print count before
print(f"mort_acc missing:         {df_clean['mort_acc'].isnull().sum()}")  # Print count before
print("-" * 40)  # Separator line

# Apply MAR Imputation Strategies
emp_mode = df_clean['emp_length_years'].mode()[0]  # Find the most frequent value (mode)
df_clean['emp_length_years'] = df_clean['emp_length_years'].fillna(emp_mode)  # Replace missing entries with the mode

mort_median = df_clean['mort_acc'].median()  # Find the middle value (median)
df_clean['mort_acc'] = df_clean['mort_acc'].fillna(mort_median)  # Replace missing entries with the median

#Print MAR missing counts AFTER imputation
print("=== MAR COUNTS AFTER ===")  # Section header
print(f"emp_length_years missing: {df_clean['emp_length_years'].isnull().sum()}")  # Verify count after (should be 0)
print(f"mort_acc missing:         {df_clean['mort_acc'].isnull().sum()}")  # Verify count after (should be 0)

=== MAR COUNTS BEFORE ===
emp_length_years missing: 180539
mort_acc missing:         260330
----------------------------------------
=== MAR COUNTS AFTER ===
emp_length_years missing: 0
mort_acc missing:         0


In [32]:
# Print MNAR missing counts BEFORE imputation ---
print("=== MNAR COUNTS BEFORE ===")  # Section header
print(f"mths_since_last_delinq missing: {df_clean['mths_since_last_delinq'].isnull().sum()}")  # Print count before
print(f"il_util_pct missing:            {df_clean['il_util_pct'].isnull().sum()}")  # Print count before
print("-" * 40)  # Separator line

#  Apply MNAR Imputation Strategies ---
# Missing means no delinquency history. Imputing with -1 as a distinct sentinel value.
df_clean['mths_since_last_delinq'] = df_clean['mths_since_last_delinq'].fillna(-1)  # Replace missing entries with -1

# Missing means no active installment loans. Imputing with 0.0 to show zero utilization.
df_clean['il_util_pct'] = df_clean['il_util_pct'].fillna(0.0)  # Replace missing entries with 0.0

# Print MNAR missing counts AFTER imputation ---
print("=== MNAR COUNTS AFTER ===")  # Section header
print(f"mths_since_last_delinq missing: {df_clean['mths_since_last_delinq'].isnull().sum()}")  # Verify count after (should be 0)
print(f"il_util_pct missing:            {df_clean['il_util_pct'].isnull().sum()}")  # Verify count after (should be 0)

=== MNAR COUNTS BEFORE ===
mths_since_last_delinq missing: 1098640
il_util_pct missing:            200167
----------------------------------------
=== MNAR COUNTS AFTER ===
mths_since_last_delinq missing: 0
il_util_pct missing:            0


##Question 1
Apply winsorisation at the 1st and 99th percentile to the six most skewed numeric columns. Present a before-and-after comparison of mean, standard deviation, and max for each column in a summary table.

In [33]:
# Identify the 6 Most Skewed Numeric Columns
numeric_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()  # Get all numerical columns

# Exclude IDs, flags, and tracking columns from skewness analysis
exclude_cols = ['loan_id', 'dirty_flag', 'issue_year']  # Define exclusion list
target_cols = [col for col in numeric_cols if col not in exclude_cols]  # Filter numerical columns

# Calculate absolute skewness and select top 6 features
skew_series = df_clean[target_cols].skew().abs().sort_values(ascending=False)  # Compute absolute skewness ranking
top_6_skewed = skew_series.head(6).index.tolist()  # Extract top 6 skewed column names
print("Top 6 Most Skewed Columns:", top_6_skewed)  # Display selected columns

Top 6 Most Skewed Columns: ['collections_12mths_fee', 'collection_recovery_fee', 'recoveries_inr', 'emi_advance_paid_inr', 'expected_loss_inr', 'avg_cur_bal_inr']


In [34]:
from scipy.stats.mstats import winsorize  # Import winsorize function

# Store Summary Statistics Before Winsorization
before_stats = df_clean[top_6_skewed].agg(['mean', 'std', 'max']).T  # Extract baseline stats

#Apply Winsorization to Top 6 Skewed Columns ---
for col in top_6_skewed:
    # limits=[0.01, 0.01] handles 1st percentile (lower) and 99th percentile (upper) bounds
    df_clean[col] = winsorize(df_clean[col], limits=[0.01, 0.01])  # Apply transformation in-place

# Store Summary Statistics After Winsorization
after_stats = df_clean[top_6_skewed].agg(['mean', 'std', 'max']).T  # Extract transformed stats

In [35]:
# Before vs after comparison
comparison_df = pd.DataFrame(index=top_6_skewed)  # Initialize reporting DataFrame

# Map values from captured statistics arrays
comparison_df['Mean (Before)'] = before_stats['mean']  # Set baseline mean
comparison_df['Mean (After)']  = after_stats['mean']   # Set transformed mean
comparison_df['Std (Before)']  = before_stats['std']    # Set baseline standard deviation
comparison_df['Std (After)']   = after_stats['std']     # Set transformed standard deviation
comparison_df['Max (Before)']  = before_stats['max']    # Set baseline maximum bound
comparison_df['Max (After)']   = after_stats['max']     # Set transformed capped maximum bound

# Display the side-by-side summary table
print("=== WINSORIZATION BEFORE-AND-AFTER SUMMARY COMPARISON ===")
print(comparison_df.round(2))

=== WINSORIZATION BEFORE-AND-AFTER SUMMARY COMPARISON ===
                         Mean (Before)  Mean (After)  Std (Before)  \
collections_12mths_fee           32.37         11.83        560.07   
collection_recovery_fee         130.71        130.71       2061.44   
recoveries_inr                   52.75         22.14        818.40   
emi_advance_paid_inr           2204.31       1708.03      13599.90   
expected_loss_inr               139.24         84.35       1285.91   
avg_cur_bal_inr                3738.27       3738.43      12551.71   

                         Std (After)  Max (Before)  Max (After)  
collections_12mths_fee         77.30     312618.52       657.22  
collection_recovery_fee      2061.44    1035421.73   1035421.73  
recoveries_inr                140.02     313742.97      1172.06  
emi_advance_paid_inr         5752.16    5329334.00     40329.10  
expected_loss_inr             500.60     189646.84      3990.42  
avg_cur_bal_inr             12551.67    2434032.72   24

#Phase 2 :Exploratory Data Analysis
Produce a bar chart and a pie chart showing the distribution of loan_status. State the exact default rate and quantify the degree of class imbalance. Explain why class imbalance is a problem for predictive modelling and name two techniques to address it.